In [1]:
import sys
sys.path.append('/home/ronedr/evolution-strategy-baselines-comparison')

In [2]:
import jax
import optax
from tqdm import tqdm
import gymnax
from evosax.problems import GymnaxProblem as Problem
from experiment.utils.nn_model import CNN
from experiment.utils.problem_utils import get_problem_settings
from experiment.run_experiments import run_experiment_permutations
from evosax.core.fitness_shaping import standardize_fitness_shaping_fn

In [3]:
action_num, out_fn = get_problem_settings("Asterix-MinAtar")
problem = Problem(
    env_name="Asterix-MinAtar",
    policy=CNN(
        num_filters=[16],
        kernel_sizes=[(5, 5)],
        strides=(1, 1),
        mlp_layer_sizes=[32, action_num]
    ),
    num_rollouts=1,
    episode_length=500
)

/home/ronedr/.local/lib/python3.11/site-packages/jax/_src/ops/scatter.py:93: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [4]:
num_generations = 5000
population_size = 256
eval_batch_size = 128
log_period = 1
seeds = list(range(0, 5))
result_dir = "../../results"
problems_gymnax = ['Asterix-MinAtar',
                   'Breakout-MinAtar',
                   'Freeway-MinAtar',
                   'SpaceInvaders-MinAtar']

In [5]:
es_dict = {
    "PGPE": {
        "optimizer": optax.adam(learning_rate=0.02),
    },
    "ASEBO": {
        "optimizer": optax.adam(learning_rate=0.01),
        "fitness_shaping_fn": standardize_fitness_shaping_fn
    },
    "Open_ES": {    
        "optimizer": optax.adam(learning_rate=0.05)
    },
    "SNES": {},
    "Sep_CMA_ES": {},
    "CMA_ES": {},
    "LES": {},
    "DES": {},
    # "EvoTF_ES": {},
}

# take only the es_algorithms we insert as arg.
running_es = es_dict

In [ ]:
for env_name in tqdm(problems_gymnax, desc="Loading Problems .."):
    try:
        action_num, out_fn = get_problem_settings(env_name)
        problem = Problem(
            env_name=env_name,
            policy=CNN(
                num_filters=[16],
                kernel_sizes=[(5, 5)],
                strides=(1, 1),
                mlp_layer_sizes=[32, action_num]
            ),
            num_rollouts=1,
            episode_length=500
        )
        print("Successfully loaded:", env_name)
        
        run_experiment_permutations(problems=[problem],
                                    es_dict=running_es,
                                    num_generations=num_generations,
                                    population_size=population_size,
                                    result_dir=result_dir, 
                                    run_again_if_exist=False,
                                    log_period=log_period,
                                    eval_batch_size=eval_batch_size,
                                    suffix_experiment_name=f"{population_size}",
                                    seeds=list(range(0, 5)))
    except Exception as e:
        print("Failed to load:", env_name, e)
        continue

Loading Problems ..:   0%|          | 0/4 [00:00<?, ?it/s]

Successfully loaded: Asterix-MinAtar



Running ES algorithms:   0%|          | 0/8 [00:00<?, ?it/s]

running the experiment ... [../../results/GymnaxProblem/Asterix-MinAtar/PGPE_256/0]
